# Nhóm 2 — CNN + MLP trên Google Colab
Notebook chỉ là adapter; model/trainer dùng chung trong `src/` và `scripts/run_group2_cnn.py`. Baseline legacy và dữ liệu thật phải được cung cấp trước khi chạy full training.

In [ ]:
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/BTL03')
PROJECT_ROOT = Path('/content/BTL03')
if not (PROJECT_ROOT / 'scripts/run_group2_cnn.py').is_file():
    raise FileNotFoundError('Checkout BTL03 vào /content/BTL03 trước khi chạy.')
import os, sys, torch
os.chdir(PROJECT_ROOT); sys.path.insert(0, str(PROJECT_ROOT))
from src.utils import seed_everything
seed_everything(42)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'KHÔNG CÓ CUDA')
if torch.cuda.is_available(): assert 'T4' in torch.cuda.get_device_name(0), 'Runtime không phải GPU T4 như yêu cầu.'

## Chuẩn bị dữ liệu
Đặt MNIST tại `data/raw/mnist/`, WIKI arrays NHWC và FER CSV numeric (2304 pixel + 7 one-hot) theo các YAML. WIKI validation/public-test và FER public-test được dùng làm validation như legacy, không phải holdout độc lập.

In [ ]:
!python scripts/download_data.py check
!python -c "from pathlib import Path; required=[Path('data/raw/wiki_gender/train_x.npy'),Path('data/raw/fer2013/fer2013_training_onehot.csv')]; missing=[str(p) for p in required if not p.is_file()]; print('Cần bổ sung dataset:', missing)"

## Kiểm tra phép xoay WIKI và huấn luyện tuần tự
MNIST giữ nguyên Flatten legacy; `pool3` là MaxPool2d gốc, hook activation map `(B,256,3,3)` rồi runner làm phẳng thành 2304 chiều. Không thay bằng GAP vì sẽ không còn parity tham số.


In [ ]:
!python scripts/run_group2_cnn.py --model mnist

In [ ]:
!python scripts/run_group2_cnn.py --model gender

In [ ]:
from IPython.display import display, Image
wiki_figure = PROJECT_ROOT / 'outputs/figures/t0204_wiki_rot90_verify.png'
if wiki_figure.exists(): display(Image(filename=str(wiki_figure)))
else: print('Runner chưa tạo ảnh WIKI; kiểm tra dữ liệu/runtime.')

In [ ]:
!python scripts/run_group2_cnn.py --model emotion

In [ ]:
import pandas as pd
from IPython.display import display
history = pd.read_csv(PROJECT_ROOT / 'experiments/csv_logs/EmotionCNN_history.csv')
display(history)
ax = history.plot(x='Epoch', y=['Train_Loss','Val_Loss'], title='FER2013: overfitting baseline (chưa có callbacks)')
ax.figure

In [ ]:
import shutil
for relative in ['outputs/checkpoints/02_cnn_mlp','outputs/features/mnist_cnn_pool3_features.npz','experiments/csv_logs','experiments/tensorboard/Group2_CNN','outputs/figures']:
    source=PROJECT_ROOT / relative
    if source.exists():
        target=DRIVE_ROOT / relative; target.parent.mkdir(parents=True,exist_ok=True)
        if source.is_dir(): shutil.copytree(source,target,dirs_exist_ok=True)
        else: shutil.copy2(source,target)
print('Đồng bộ artifacts hiện có lên Drive:', DRIVE_ROOT)